In [25]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
import sys, os

# __file__ doesn't exist in notebooks, so derive path from the known notebook location
notebooks_dir = os.path.join(os.path.abspath('.'), 'notebooks')
sys.path.insert(0, notebooks_dir)

from utils.sampling import load_or_sample


df = load_or_sample(
    filepath="/home/ishola/codetopia/mga/raw-data/openfoodfacts.csv",
    output_csv="/home/ishola/codetopia/mga/raw-data/openfoodfacts_sample.csv",
    output_meta="/home/ishola/codetopia/mga/raw-data/openfoodfacts_sample_meta.json",
    sample_size=1_000_000,
    seed=42,
    chunk_size=25_000,
)

✅ Cached sample found — loading from disk.
   Shape: (1000000, 134)
   🧠 Memory: 1597 MB


In [27]:

# df['created_datetime'] = pd.to_datetime(df['created_datetime'], errors='coerce')

# yearly_counts = df['created_datetime'].dt.year.value_counts().sort_index()

# fig, ax = plt.subplots(figsize=(12, 6))
# bars = ax.bar(yearly_counts.index.astype(int), yearly_counts.values,
#               color='#6366f1', edgecolor='#4f46e5', linewidth=0.8)

# # Annotate bars
# for bar in bars:
#     height = bar.get_height()
#     ax.text(bar.get_x() + bar.get_width() / 2, height,
#             f'{int(height):,}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# ax.set_title('Product Counts by Year', fontsize=16, fontweight='bold', pad=15)
# ax.set_xlabel('Year', fontsize=12)
# ax.set_ylabel('Number of Products', fontsize=12)
# ax.set_xticks(yearly_counts.index.astype(int))
# ax.set_xticklabels(yearly_counts.index.astype(int), rotation=45, ha='right')
# ax.grid(axis='y', alpha=0.3, linestyle='--')
# ax.spines[['top', 'right']].set_visible(False)

# plt.tight_layout()
# plt.show()


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Columns: 134 entries, code to water_100g
dtypes: float64(123), object(11)
memory usage: 1022.3+ MB


In [29]:
# 1. Calculate missing ratio for each column
missing_ratio = df.isnull().mean()

# 2. Print columns and their missing ratios
print("Missing Ratios per Column:")
print(missing_ratio)

# 3. Identify columns with <= 80% missingness (keep these)
cols_to_keep = missing_ratio[missing_ratio <= 0.80].index


print(f'\nColumns to keep: \n {cols_to_keep}')
# 4. Filter the DataFrame
df_without_missing = df[cols_to_keep]

# # Optional: Verify the new shape
print(f"\nCount of columns to keep: {df_without_missing.shape[1]}")

Missing Ratios per Column:
code                        0.000000
product_name                0.073834
brands_tags                 0.373546
categories_tags             0.584226
origins_tags                0.961399
                              ...   
sulphate_100g               0.999872
nitrate_100g                0.999923
acidity_100g                0.999988
carbohydrates-total_100g    0.999687
water_100g                  0.944996
Length: 134, dtype: float64

Columns to keep: 
 Index(['code', 'product_name', 'brands_tags', 'categories_tags', 'labels_tags',
       'countries_tags', 'ingredients_tags', 'ingredients_analysis_tags',
       'nova_group', 'main_category_en', 'energy-kj_100g', 'energy-kcal_100g',
       'energy_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates_100g',
       'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g',
       'sodium_100g'],
      dtype='object')

Count of columns to keep: 21


In [30]:
# Dropping columns with 0 variance

# 1. Identify numeric columns with 0 variance
numeric_cols = df_without_missing.select_dtypes(include=['number']).columns
zero_var_numeric = [col for col in numeric_cols if df_without_missing[col].var() == 0]

# 2. Identify categorical columns with only 1 unique value
categorical_cols = df_without_missing.select_dtypes(exclude=['number']).columns
zero_var_categorical = [col for col in categorical_cols if df_without_missing[col].nunique() == 1]

# 3. Drop both sets
cols_to_drop = zero_var_numeric + zero_var_categorical
df_with_variance = df_without_missing.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} zero-variance columns: {cols_to_drop}")
print(f"Remaining shape: {df_without_missing.shape}")


Dropped 0 zero-variance columns: []
Remaining shape: (1000000, 21)


In [31]:
# ── Domain-aware redundancy removal 
safe_to_drop = [
    'energy_100g',      # duplicate of energy-kj_100g 
    'energy-kj_100g',   # duplicate of energy-kj_100g 
    'sodium_100g',      # = salt × 0.4 (deterministic)
]

safe_to_drop = [c for c in safe_to_drop if c in df_with_variance.columns]
df_no_redundancy = df_with_variance.drop(columns=safe_to_drop)

print(f"Dropped {len(safe_to_drop)} unit-duplicate columns: {safe_to_drop}")
print(f"Remaining shape: {df_no_redundancy.shape}")
print(f"\nRemaining columns:\n{df_no_redundancy.columns.tolist()}")


Dropped 3 unit-duplicate columns: ['energy_100g', 'energy-kj_100g', 'sodium_100g']
Remaining shape: (1000000, 18)

Remaining columns:
['code', 'product_name', 'brands_tags', 'categories_tags', 'labels_tags', 'countries_tags', 'ingredients_tags', 'ingredients_analysis_tags', 'nova_group', 'main_category_en', 'energy-kcal_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g']


In [32]:
## remove duplicated rows
df_no_redundancy.drop_duplicates(inplace=True, ignore_index=True)
df_no_redundancy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   code                       1000000 non-null  object 
 1   product_name               926166 non-null   object 
 2   brands_tags                626454 non-null   object 
 3   categories_tags            415774 non-null   object 
 4   labels_tags                274062 non-null   object 
 5   countries_tags             994901 non-null   object 
 6   ingredients_tags           288723 non-null   object 
 7   ingredients_analysis_tags  305571 non-null   object 
 8   nova_group                 256704 non-null   float64
 9   main_category_en           415772 non-null   object 
 10  energy-kcal_100g           496681 non-null   float64
 11  fat_100g                   491522 non-null   float64
 12  saturated-fat_100g         475323 non-null   float64
 13  carbohydrates

In [33]:
# Returns the total number of rows that with duplicated barcodes(same product)
num_duplicates = df_no_redundancy['code'].duplicated().sum()
print(num_duplicates)

4


In [34]:
# Keep the row with the most non-null values for each barcode
df_no_redundancy['_completeness'] = df_no_redundancy.notna().sum(axis=1)
df_no_redundancy = (
    df_no_redundancy
    .sort_values('_completeness', ascending=False)
    .drop_duplicates(subset='code', keep='first')
    .drop(columns='_completeness')
)

print(f"After dedup: {len(df_no_redundancy):,} unique products")

After dedup: 999,996 unique products


In [35]:
# A row is useful if it has a category AND at least one nutrition value
nutrition_cols = ['energy-kcal_100g', 'fat_100g', 'saturated-fat_100g',
                  'carbohydrates_100g', 'sugars_100g', 'fiber_100g',
                  'proteins_100g', 'salt_100g']

has_category = df_no_redundancy['main_category_en'].notna() | df_no_redundancy['categories_tags'].notna()
has_nutrition = df_no_redundancy[nutrition_cols].notna().any(axis=1)

df_clean = df_no_redundancy[has_category & has_nutrition].copy()

print(f"Removed {len(df_no_redundancy) - len(df_clean):,} rows with no category and no nutrition")
print(f"Remaining: {len(df_clean):,} rows ({len(df_clean)/len(df_no_redundancy)*100:.1f}%)")


Removed 752,229 rows with no category and no nutrition
Remaining: 247,767 rows (24.8%)


In [36]:
df_clean.isna().mean()

code                         0.000000
product_name                 0.019522
brands_tags                  0.177062
categories_tags              0.000000
labels_tags                  0.492886
countries_tags               0.000319
ingredients_tags             0.451194
ingredients_analysis_tags    0.435675
nova_group                   0.467697
main_category_en             0.000000
energy-kcal_100g             0.011725
fat_100g                     0.017646
saturated-fat_100g           0.030993
carbohydrates_100g           0.018162
sugars_100g                  0.026384
fiber_100g                   0.465441
proteins_100g                0.017484
salt_100g                    0.066054
dtype: float64

In [37]:
# Nutrient values cannot exceed 100g per 100g of product
# Energy can be up to ~900 kcal/100g (pure fat), so use a generous cap
caps = {
    'fat_100g': 100, 'saturated-fat_100g': 100,
    'carbohydrates_100g': 100, 'sugars_100g': 100,
    'fiber_100g': 100, 'proteins_100g': 100,
    'salt_100g': 100,
    'energy-kcal_100g': 900,  # ~pure fat
}

for col, cap in caps.items():
    if col in df_clean.columns:
        bad = (df_clean[col] < 0) | (df_clean[col] > cap)
        df_clean.loc[bad, col] = None  # Null out, don't drop the row
        if bad.sum() > 0:
            print(f"  Nulled {bad.sum():,} impossible values in {col}")


  Nulled 62 impossible values in fat_100g
  Nulled 33 impossible values in saturated-fat_100g
  Nulled 131 impossible values in carbohydrates_100g
  Nulled 46 impossible values in sugars_100g
  Nulled 43 impossible values in fiber_100g
  Nulled 33 impossible values in proteins_100g
  Nulled 100 impossible values in salt_100g
  Nulled 380 impossible values in energy-kcal_100g


In [38]:
### impossible macros combination. macros shouldn't add up to 100g for 100g serving
macro_cols = ['fat_100g', 'carbohydrates_100g', 'proteins_100g', 'fiber_100g']
present = [c for c in macro_cols if c in df_clean.columns]

df_clean['_macro_sum'] = df_clean[present].sum(axis=1, min_count=len(present))
corrupt = df_clean['_macro_sum'] > 104  # 4g tolerance for rounding

df_clean.loc[corrupt, present] = None   # Null the values, keep the row for category signals
df_clean = df_clean.drop(columns='_macro_sum')

print(f"Nulled macros for {corrupt.sum():,} rows with impossible composition")


Nulled macros for 1,441 rows with impossible composition


In [39]:
#  NOVA Group Validation 
# NOVA scale:
#   1 = Unprocessed / minimally processed (fresh fruit, eggs, meat)
#   2 = Processed culinary ingredients (oil, flour, sugar, salt)
#   3 = Processed foods (canned veg, cheese, cured meats)
#   4 = Ultra-processed (soft drinks, packaged snacks, ready meals)

valid_nova = {1.0, 2.0, 3.0, 4.0}

# Check current state
print("NOVA group value counts (including invalids):")
print(df_clean['nova_group'].value_counts(dropna=False).sort_index())

# Identify invalid entries
invalid_nova = df_clean['nova_group'].notna() & ~df_clean['nova_group'].isin(valid_nova)
print(f"\nInvalid NOVA values: {invalid_nova.sum():,}")

NOVA group value counts (including invalids):
nova_group
1.0     18250
2.0      8064
3.0     29119
4.0     76454
NaN    115880
Name: count, dtype: int64

Invalid NOVA values: 0


In [40]:
df_clean['main_category_en'] = (
    df_clean['main_category_en']
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
)
print(f"Unique categories: {df_clean['main_category_en'].nunique():,}")

Unique categories: 20,655


In [41]:
# Acceptance criteria explicitly calls out these three
critical_cols = ['sugars_100g', 'proteins_100g', 'product_name']
# Decision: Don't drop — null out and flag. A product with no sugar
# value can still contribute to category-level protein analysis.
# ONLY drop if ALL three are null simultaneously.
all_critical_null = df_clean[critical_cols].isna().all(axis=1)
df_clean = df_clean[~all_critical_null].copy()
print(f"Dropped {all_critical_null.sum():,} rows with ALL critical fields null")

Dropped 79 rows with ALL critical fields null


In [42]:
# This is your deliverable for Story 1
df_clean.to_csv(
    "/home/ishola/codetopia/mga/raw-data/openfoodfacts_clean.csv",
    index=False
)
print(f"Story 1 deliverable saved. Shape: {df_clean.shape}")

Story 1 deliverable saved. Shape: (247688, 18)


In [43]:
# ── Step 1: Parse categories_tags into a list ────────────────
df_clean['categories_list'] = (
    df_clean['categories_tags']
    .str.split(',')
    .apply(lambda x: [t.strip().replace('en:', '').replace('-', ' ')
                      for t in x] if isinstance(x, list) else [])
)

# ── Step 2: Filter to snacks — DON'T overwrite categories_list ──
SNACK_KEYWORDS = [
    'snack', 'biscuit', 'cookie', 'cracker', 'chip', 'crisp',
    'bar', 'chocolate', 'candy', 'confection', 'wafer', 'pretzel',
    'popcorn', 'nut', 'seed', 'trail mix', 'granola', 'cereal bar',
    'rice cake', 'puff', 'corn snack', 'tortilla', 'nacho',
    'protein bar', 'energy bar', 'fruit snack', 'dried fruit',
]

def is_snack(cat_list):
    return any(kw in tag for tag in cat_list for kw in SNACK_KEYWORDS)

# Use boolean mask to FILTER rows, not overwrite the column
snack_mask = df_clean['categories_list'].apply(is_snack)
df_snacks = df_clean[snack_mask].copy()

print(f"Snack products: {len(df_snacks):,} ({len(df_snacks)/len(df_clean)*100:.1f}%)")

# ── Step 3: Assign sub-categories ───────────────────────────
SNACK_SUBCATEGORY_MAP = {
    'Protein Bars & Supplements': ['protein bar', 'protein snack', 'energy bar',
                                    'sport bar', 'fitness bar', 'whey'],
    'Nuts, Seeds & Trail Mix':    ['nut', 'seed', 'trail mix', 'almond', 'cashew',
                                    'peanut', 'pistachio', 'walnut'],
    'Chips & Crisps':             ['chip', 'crisp', 'popcorn', 'pretzel', 'puff',
                                    'tortilla', 'corn snack', 'nacho'],
    'Cookies & Biscuits':         ['cookie', 'biscuit', 'wafer', 'shortbread',
                                    'digestive', 'graham'],
    'Chocolate & Confections':    ['chocolate', 'candy', 'sweet', 'confection',
                                    'caramel', 'fudge', 'gummy', 'jelly'],
    'Cereal & Granola Bars':      ['granola bar', 'cereal bar', 'oat bar',
                                    'muesli bar', 'rice bar', 'rice cake'],
    'Fruit Snacks & Dried Fruit': ['fruit snack', 'dried fruit', 'raisin',
                                    'date', 'apricot', 'fig snack'],
    'Crackers & Rice Cakes':      ['cracker', 'rice cake', 'breadstick',
                                    'rusk', 'crispbread'],
}

def assign_snack_subcategory(cat_list):
    if not isinstance(cat_list, list) or len(cat_list) == 0:
        return 'Other Snacks'
    cats_lower = ' '.join(cat_list).lower()
    for bucket, keywords in SNACK_SUBCATEGORY_MAP.items():
        if any(kw in cats_lower for kw in keywords):
            return bucket
    return 'Other Snacks'

df_snacks['primary_category'] = df_snacks['categories_list'].apply(
    assign_snack_subcategory
)

print(df_snacks['primary_category'].value_counts())
other_pct = (df_snacks['primary_category'] == 'Other Snacks').mean() * 100
print(f"\n'Other Snacks': {other_pct:.1f}% — add keywords if > 20%")


Snack products: 66,205 (26.7%)
primary_category
Chocolate & Confections       21676
Nuts, Seeds & Trail Mix       15777
Cookies & Biscuits            14793
Chips & Crisps                 6753
Other Snacks                   4159
Fruit Snacks & Dried Fruit     1956
Protein Bars & Supplements      976
Crackers & Rice Cakes           107
Cereal & Granola Bars             8
Name: count, dtype: int64

'Other Snacks': 6.3% — add keywords if > 20%


In [45]:
import plotly.express as px
import plotly.graph_objects as go

# ── 3.1 Category-level aggregates ───────────────────────────
category_stats = (
    df_snacks
    .groupby('primary_category')
    .agg(
        median_sugar=('sugars_100g', 'median'),
        median_protein=('proteins_100g', 'median'),
        median_fat=('fat_100g', 'median'),
        median_fiber=('fiber_100g', 'median'),
        product_count=('code', 'count'),
    )
    .reset_index()
    .dropna(subset=['median_sugar', 'median_protein'])
)

print(category_stats.sort_values('product_count', ascending=False))


             primary_category  median_sugar  median_protein  median_fat  \
2     Chocolate & Confections     41.000000            6.00        18.3   
6     Nuts, Seeds & Trail Mix      4.000000            9.10        29.0   
3          Cookies & Biscuits     27.300000            6.10        20.0   
1              Chips & Crisps      2.000000            6.50        23.0   
7                Other Snacks      3.500000            7.00        12.2   
5  Fruit Snacks & Dried Fruit     53.100000            2.50         0.5   
8  Protein Bars & Supplements      9.600000           29.60        14.0   
4       Crackers & Rice Cakes      2.600000           11.00        12.0   
0       Cereal & Granola Bars     17.916667            6.65        13.6   

   median_fiber  product_count  
2      2.300000          21676  
6      5.200000          15777  
3      2.618000          14793  
1      3.900000           6753  
7      2.181957           4159  
5      7.000000           1956  
8      6.932806   

In [46]:
# ── 3.2 Scatter: Sugar vs Protein per sub-category ──────────
overall_median_sugar   = df_snacks['sugars_100g'].median()
overall_median_protein = df_snacks['proteins_100g'].median()

fig = px.scatter(
    category_stats,
    x='median_sugar',
    y='median_protein',
    size='product_count',
    color='primary_category',
    text='primary_category',
    hover_data={
        'median_fat': ':.1f',
        'median_fiber': ':.1f',
        'product_count': ':,',
        'primary_category': False,
    },
    title='Snack Market Nutrient Matrix<br>'
          '<sup>Bubble size = number of products | '
          'Top-left = High Protein, Low Sugar (the Blue Ocean)</sup>',
    labels={
        'median_sugar':   'Median Sugar (g/100g)',
        'median_protein': 'Median Protein (g/100g)',
        'primary_category': 'Snack Category',
    },
    template='plotly_white',
)

# ── Reference lines: overall medians ────────────────────────
fig.add_hline(
    y=overall_median_protein,
    line_dash='dash', line_color='green', line_width=1,
    annotation_text='Median protein',
    annotation_position='top right',
)
fig.add_vline(
    x=overall_median_sugar,
    line_dash='dash', line_color='red', line_width=1,
    annotation_text='Median sugar',
    annotation_position='top right',
)

# ── Label the empty quadrant ─────────────────────────────────
fig.add_annotation(
    x=overall_median_sugar * 0.3,
    y=overall_median_protein * 1.8,
    text="🎯 Blue Ocean<br>(High Protein, Low Sugar)",
    showarrow=False,
    bgcolor='rgba(0,200,100,0.15)',
    bordercolor='green',
    borderwidth=1,
    font=dict(color='green', size=12),
)

fig.update_traces(textposition='top center', textfont_size=10)
fig.update_layout(height=600, showlegend=False)
fig.show()


In [47]:
# ── 4.1 Define "healthy" threshold ──────────────────────────
# High protein = above overall median protein for snacks
# Low sugar   = below overall median sugar for snacks
high_protein_threshold = df_snacks['proteins_100g'].median()
low_sugar_threshold    = df_snacks['sugars_100g'].median()

print(f"High protein threshold: > {high_protein_threshold:.1f}g/100g")
print(f"Low sugar threshold:    < {low_sugar_threshold:.1f}g/100g")

# ── 4.2 Count healthy products per sub-category ─────────────
def gap_stats(group):
    has_nutrition = group[['proteins_100g', 'sugars_100g']].notna().all(axis=1)
    measurable = group[has_nutrition]
    if len(measurable) == 0:
        return None
    healthy = measurable[
        (measurable['proteins_100g'] > high_protein_threshold) &
        (measurable['sugars_100g'] < low_sugar_threshold)
    ]
    return {
        'total_products': len(group),
        'measurable_products': len(measurable),
        'healthy_products': len(healthy),
        'gap_ratio': len(healthy) / len(measurable),  # % already healthy
        'gap_pct_missing': 1 - (len(healthy) / len(measurable)),  # % NOT healthy
        'avg_protein_healthy': healthy['proteins_100g'].mean(),
        'avg_sugar_healthy':   healthy['sugars_100g'].mean(),
    }

gap_df = (
    df_snacks
    .groupby('primary_category')
    .apply(gap_stats)
    .dropna()
    .apply(pd.Series)
    .sort_values('gap_ratio')   # lowest = biggest opportunity
    .reset_index()
)

print("\n── Market Gap Analysis ──────────────────────────────────")
print(gap_df[['primary_category', 'total_products', 'healthy_products',
               'gap_ratio', 'gap_pct_missing']].to_string(index=False))


High protein threshold: > 6.5g/100g
Low sugar threshold:    < 21.6g/100g

── Market Gap Analysis ──────────────────────────────────
          primary_category  total_products  healthy_products  gap_ratio  gap_pct_missing
Fruit Snacks & Dried Fruit          1956.0              67.0   0.035171         0.964829
   Chocolate & Confections         21676.0            2765.0   0.130869         0.869131
        Cookies & Biscuits         14793.0            2528.0   0.173817         0.826183
            Chips & Crisps          6753.0            3027.0   0.461855         0.538145
              Other Snacks          4159.0            1991.0   0.493922         0.506078
   Nuts, Seeds & Trail Mix         15777.0            8404.0   0.557997         0.442003
     Cereal & Granola Bars             8.0               4.0   0.571429         0.428571
Protein Bars & Supplements           976.0             617.0   0.659188         0.340812
     Crackers & Rice Cakes           107.0              86.0   0.84

/tmp/ipykernel_86392/3595371346.py:33: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [49]:
# ── 4.3 Auto-generate the recommendation text ───────────────
top_gap = gap_df.iloc[0]  # lowest gap_ratio = biggest opportunity

target = df_snacks[
    (df_snacks['primary_category'] == top_gap['primary_category']) &
    (df_snacks['proteins_100g'] > high_protein_threshold) &
    (df_snacks['sugars_100g'] < low_sugar_threshold)
]

target_protein = round(target['proteins_100g'].quantile(0.75), 1)
target_sugar   = round(target['sugars_100g'].quantile(0.25), 1)

print(f"""
╔══════════════════════════════════════════════════════════════╗
║                    KEY INSIGHT                              ║
╠══════════════════════════════════════════════════════════════╣
  Based on the data, the biggest market opportunity is in

  ➡  {top_gap['primary_category'].upper()}

  Only {top_gap['gap_ratio']*100:.1f}% of products in this category are
  high-protein AND low-sugar — meaning {top_gap['gap_pct_missing']*100:.1f}% of the
  {int(top_gap['total_products']):,} products in this segment FAIL the health bar.

  A new product targeting:
    • ≥ {target_protein}g protein per 100g
    • < {target_sugar}g sugar per 100g

  ...would be in the top quartile of healthy snacks in this
  category and face minimal direct competition.
╚══════════════════════════════════════════════════════════════╝
""")



╔══════════════════════════════════════════════════════════════╗
║                    KEY INSIGHT                              ║
╠══════════════════════════════════════════════════════════════╣
  Based on the data, the biggest market opportunity is in

  ➡  FRUIT SNACKS & DRIED FRUIT

  Only 3.5% of products in this category are
  high-protein AND low-sugar — meaning 96.5% of the
  1,956 products in this segment FAIL the health bar.

  A new product targeting:
    • ≥ 22.2g protein per 100g
    • < 3.2g sugar per 100g

  ...would be in the top quartile of healthy snacks in this
  category and face minimal direct competition.
╚══════════════════════════════════════════════════════════════╝

